# THINGS Database — Candidate Filter

Filters THINGS/THINGSplus object concepts by memorability, recognizability, holdability,
pleasantness, and arousal, and saves the result as `results.csv` (the expected input to
`00_prepare_candidates.ipynb`).

Expected layout after `osf -p jum2f clone THINGS-database`:

```
THINGS-database/
├── 01_image-level/
│   └── _images-metadata_things.tsv
├── 02_object-level/
│   ├── _concepts-metadata_things.tsv
│   └── _property-ratings.tsv
└── 03_category-level/
    └── category53_wide-format.tsv
```

Property scales (THINGSplus):

| column                    | scale | meaning                          |
|---------------------------|-------|-----------------------------------|
| `memorability_cr`         | 0–1   | image hit rate (concept-averaged) |
| `recognizability`         | 0–1   | 0 = not recognizable, 1 = fully   |
| `property_hold_mean`      | 1–7   | 7 = very easy to hold in one hand |
| `property_pleasant_mean`  | 1–7   | 7 = very pleasant                 |
| `property_arousal_mean`   | 1–7   | 7 = very arousing                 |


In [101]:
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

logging.basicConfig(format="  %(levelname)s  %(message)s", level=logging.INFO)
log = logging.getLogger(__name__)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", 32)
pd.set_option("display.max_rows", 60)


## Column names & display config

In [102]:
# ── Raw column names (as they appear in the TSV files) ───────────────────────

class Col:
    MEM        = "memorability_cr"
    REC        = "recognizability"
    HOLD       = "property_hold_mean"
    HOLD_SD    = "property_hold_SD"
    PLEAS      = "property_pleasant_mean"
    PLEAS_SD   = "property_pleasant_SD"
    SIZE       = "size_mean"
    MANMADE    = "property_manmade_mean"
    MANMADE_SD = "property_manmade_SD"
    GRASP      = "property_grasp_mean"
    GRASP_SD   = "property_grasp_SD"
    NATURAL    = "property_natural_mean"
    NATURAL_SD = "property_natural_SD"
    AROUSAL    = "property_arousal_mean"
    AROUSAL_SD = "property_arousal_SD"
    NAMEABILITY = "nameability"
    NAMEABILITY_SE = "nameability_SE"
    SUBTLEX_FREQ = "SUBTLEX freq"
    ZIPF_SUBTLEX = "ZipfSUBTLEX_en"
    WORD = "Word"

PROPERTY_COLS = (
    Col.HOLD, Col.HOLD_SD,
    Col.PLEAS, Col.PLEAS_SD,
    Col.SIZE,
    Col.MANMADE, Col.MANMADE_SD,
    Col.GRASP, Col.GRASP_SD,
    Col.NATURAL, Col.NATURAL_SD,
    Col.AROUSAL, Col.AROUSAL_SD
)

IMAGE_COLS = [Col.MEM, Col.REC, Col.NAMEABILITY, Col.NAMEABILITY_SE]

DISPLAY_COLS = [
    "Word", "categories_53", Col.SIZE,
    Col.MEM, Col.REC, Col.HOLD, Col.HOLD_SD, Col.PLEAS, Col.PLEAS_SD, Col.AROUSAL, Col.AROUSAL_SD,
    Col.MANMADE, Col.MANMADE_SD, Col.GRASP, Col.GRASP_SD, Col.NATURAL, Col.NATURAL_SD,
    Col.NAMEABILITY, Col.NAMEABILITY_SE, Col.SUBTLEX_FREQ, Col.ZIPF_SUBTLEX
]

# All columns included in descriptive statistics (means paired with their SDs)
STAT_COLS = [
    Col.MEM,
    Col.REC,
    Col.HOLD,    Col.HOLD_SD,
    Col.PLEAS,   Col.PLEAS_SD,
    Col.AROUSAL, Col.AROUSAL_SD,
    Col.MANMADE, Col.MANMADE_SD,
    Col.GRASP,   Col.GRASP_SD,
    Col.NATURAL, Col.NATURAL_SD,
    Col.SIZE,
    Col.NAMEABILITY, Col.NAMEABILITY_SE,
    Col.SUBTLEX_FREQ, Col.ZIPF_SUBTLEX
]


## Filter settings

Ranges are `(min, max)`;
use `None` for "no limit" on either side.


In [103]:
# ── Filter settings (defaults requested) ──────────────────────────────────────

CATEGORIES: Optional[list[str]] = None   # e.g. ["food", "tool"] to restrict; None = all

MEM_RANGE      = (0.7, 1.0)   # memorability_cr
REC_RANGE      = (0.7, 1.0)   # recognizability
HOLD_RANGE     = (5.0, 7.0)   # property_hold_mean
PLEAS_RANGE    = (4.0, 6.0)   # property_pleasant_mean
AROUSAL_RANGE  = (1.0, 4.5)   # property_arousal_mean

N_PER_CATEGORY: Optional[int] = None     # cap concepts per category; None = unlimited
SORT_BY        = Col.MEM                 # one of: memorability_cr / recognizability /
                                          # property_hold_mean / property_pleasant_mean /
                                          # property_arousal_mean / "category"

DB_ROOT_HINT: Optional[str] = None       # set explicitly if auto-discovery below fails,
                                          # e.g. "/path/to/THINGS-database"
OUTPUT_PATH = "results.csv"             # feeds directly into 00_prepare_candidates.ipynb


@dataclass
class FilterParams:
    categories:     Optional[list[str]] = None
    mem_range:      tuple = (None, None)
    rec_range:      tuple = (None, None)
    hold_range:     tuple = (None, None)
    pleas_range:    tuple = (None, None)
    arous_range:    tuple = (None, None)
    n_per_category: Optional[int] = None
    sort_by:        str = field(default=Col.MEM)
    output:         Optional[str] = None


params = FilterParams(
    categories     = CATEGORIES,
    mem_range      = MEM_RANGE,
    rec_range      = REC_RANGE,
    hold_range     = HOLD_RANGE,
    pleas_range    = PLEAS_RANGE,
    arous_range    = AROUSAL_RANGE,
    n_per_category = N_PER_CATEGORY,
    sort_by        = SORT_BY,
    output         = OUTPUT_PATH,
)
params


FilterParams(categories=None, mem_range=(0.7, 1.0), rec_range=(0.7, 1.0), hold_range=(5.0, 7.0), pleas_range=(4.0, 6.0), arous_range=(1.0, 4.5), n_per_category=None, sort_by='memorability_cr', output='results.csv')

## Database discovery & I/O helpers

In [104]:
_SEARCH_ROOTS = [
    Path("."),
    Path("./things_database"),
    Path("/home/hivrim8h/projects/things_database"),
    Path.home() / "Desktop" / "things-database",
]

_UNIQUE_ID = "uniqueID"


def find_db_root(hint: Optional[str] = None) -> Path:
    """Return the THINGS-database root that contains 01_image-level/."""
    candidates = ([Path(hint)] if hint else []) + _SEARCH_ROOTS
    for p in candidates:
        if (p / "01_image-level").exists():
            return p.resolve()
    raise FileNotFoundError(
        "Cannot find THINGS-database root.\n"
        "Set DB_ROOT_HINT above to /path/to/THINGS-database."
    )


def _read_tsv(path: Path) -> pd.DataFrame:
    """Read a TSV (falls back to CSV if only one column is parsed)."""
    df = pd.read_csv(path, sep="\t", low_memory=False)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=",", low_memory=False)
    df.columns = df.columns.str.strip()
    return df


def _merge_key(left: pd.DataFrame, right: pd.DataFrame) -> str:
    """Prefer uniqueID as join key, fall back to Word."""
    if _UNIQUE_ID in left.columns and _UNIQUE_ID in right.columns:
        return _UNIQUE_ID
    return "Word"


## Loaders

In [105]:
def load_concepts(root: Path) -> pd.DataFrame:
    path = root / "02_object-level" / "_concepts-metadata_things.tsv"
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    df = _read_tsv(path)
    df.rename(columns={"word": "Word", "Concept": "Word"}, inplace=True)

    if "Word" not in df.columns:
        first_str_col = df.select_dtypes("object").columns[0]
        df.rename(columns={first_str_col: "Word"}, inplace=True)

    log.info("Concepts: %d rows | columns: %s …", len(df), list(df.columns[:8]))
    return df


def load_property_ratings(root: Path, concepts: pd.DataFrame) -> pd.DataFrame:
    path = root / "02_object-level" / "_property-ratings.tsv"

    df  = _read_tsv(path)
    key = _merge_key(concepts, df)

    log.info("Property TSV raw columns: %s", list(df.columns))
    # Match TSV headers to PROPERTY_COLS case-insensitively, then rename to the
    # canonical Col name so the rest of the code can rely on exact spelling.
    tsv_lower = {c.lower(): c for c in df.columns if c != key}
    rename    = {tsv_lower[p.lower()]: p for p in PROPERTY_COLS if p.lower() in tsv_lower}
    df.rename(columns=rename, inplace=True)

    present = [c for c in PROPERTY_COLS if c in df.columns]

    return concepts.merge(df[[key] + present], on=key, how="left")


def load_image_level(root: Path, concepts: pd.DataFrame) -> pd.DataFrame:
    path = root / "01_image-level" / "_images-metadata_things.tsv"

    df = _read_tsv(path)
    key = _merge_key(concepts, df)

    tsv_lower = {c.lower(): c for c in df.columns if c != key}
    rename    = {tsv_lower[p.lower()]: p for p in IMAGE_COLS if p.lower() in tsv_lower}
    df.rename(columns=rename, inplace=True)

    present = [c for c in IMAGE_COLS if c in df.columns]

    return concepts.merge(df[[key] + present], on=key, how="left")


def load_categories(root: Path, concepts: pd.DataFrame) -> pd.DataFrame:
    """
    Load the 53-category membership matrix and attach two columns:
      categories_53  – list of category names the concept belongs to
      cat53_string   – semicolon-joined string for easy filtering/display
    """
    path = root / "03_category-level" / "category53_wide-format.tsv"

    df = _read_tsv(path)

    id_col_names = {_UNIQUE_ID.lower(), "word", "concept", "object", "index"}
    id_cols  = [c for c in df.columns if c.lower() in id_col_names]
    cat_cols = [c for c in df.columns if c not in id_cols]

    key = _merge_key(concepts, df) if id_cols else None
    if key:
        concepts = concepts.merge(df[[key] + cat_cols], on=key, how="left")
    else:
        n_rows = min(len(df), len(concepts))
        for col in cat_cols:
            concepts.loc[:n_rows - 1, col] = df[col].values[:n_rows]

    concepts["categories_53"] = concepts.apply(
        lambda row: [c for c in cat_cols if row.get(c, 0) == 1], axis=1
    )
    concepts["cat53_string"] = concepts["categories_53"].apply("; ".join)
    log.info("53-category memberships attached | %d categories.", len(cat_cols))
    return concepts


def load_all(root: Path) -> pd.DataFrame:
    """Load and join all THINGS data sources into one DataFrame."""
    concepts = load_concepts(root)
    concepts = load_property_ratings(root, concepts)
    concepts = load_image_level(root, concepts)
    concepts = load_categories(root, concepts)
    return concepts


## Filtering

In [106]:
def _range_mask(series: pd.Series, lo: Optional[float], hi: Optional[float]) -> pd.Series:
    mask = pd.Series(True, index=series.index)
    if lo is not None:
        mask &= series.fillna(-np.inf) >= lo
    if hi is not None:
        mask &= series.fillna(np.inf)  <= hi
    return mask


def filter_concepts(df: pd.DataFrame, params: FilterParams) -> pd.DataFrame:
    result = df.copy()

    # ── Category ─────────────────────────────────────────────────────────────
    if params.categories:
        if "cat53_string" not in result.columns:
            log.warning("cat53_string column not found — skipping category filter.")
        else:
            cats_lower = [c.lower().strip() for c in params.categories]
            mask       = result["cat53_string"].str.lower().apply(
                lambda s: any(c in s for c in cats_lower)
            )
            before, result = len(result), result[mask]
            log.info("Category filter → %d/%d kept", len(result), before)

    # ── Numeric ranges ────────────────────────────────────────────────────────
    for col, rng, label in [
        (Col.MEM,   params.mem_range,   "Memorability"),
        (Col.REC,   params.rec_range,   "Recognizability"),
        (Col.HOLD,  params.hold_range,  "Holdability"),
        (Col.PLEAS, params.pleas_range, "Pleasantness"),
        (Col.AROUSAL, params.arous_range, "Arousal"),
    ]:
        if rng == (None, None):
            continue
        if col not in result.columns:
            log.warning("'%s' missing — skipping %s filter.", col, label)
            continue
        before, result = len(result), result[_range_mask(result[col], *rng)]
        log.info("%s (%s) %s → %d/%d kept", label, col, rng, len(result), before)

    # ── Stratified sampling ───────────────────────────────────────────────────
    if params.n_per_category and "categories_53" in result.columns:
        uid_col = _UNIQUE_ID if _UNIQUE_ID in result.columns else result.columns[0]
        sampled = (result.explode("categories_53")
                         .groupby("categories_53", group_keys=False)
                         .head(params.n_per_category))
        result  = sampled.drop_duplicates(subset=[uid_col])
        log.info("Stratified sample (≤%d/category) → %d unique concepts",
                 params.n_per_category, len(result))

    # ── Sort ──────────────────────────────────────────────────────────────────
    if params.sort_by == "category":
        # Explode so each concept appears once per category, sort alphabetically
        # on category name, then keep only the first (alphabetically earliest)
        # category row for each concept to avoid duplicates.
        uid_col = _UNIQUE_ID if _UNIQUE_ID in result.columns else result.columns[0]
        result = (result.explode("categories_53")
                        .sort_values(["categories_53", "Word"])
                        .drop_duplicates(subset=[uid_col])
                        .dropna(subset=["categories_53"])
                        .reset_index(drop=True))
    elif params.sort_by in result.columns:
        result = result.sort_values(params.sort_by, ascending=False, na_position="last")

    return result.reset_index(drop=True)


## Report

## Run it

In [107]:
root = find_db_root(DB_ROOT_HINT)
log.info("THINGS-database root: %s", root)

concepts = load_all(root)


  INFO  THINGS-database root: /home/hivrim8h/projects/things_database
  INFO  Concepts: 1854 rows | columns: ['Word', 'uniqueID', 'WordNet ID', 'Wordnet ID2', 'Wordnet ID3', 'Wordnet ID4', 'WordNet Synonyms', 'Definition (from WordNet, Google, or Wikipedia)'] …
  INFO  Property TSV raw columns: ['Unnamed: 0', 'Word', 'uniqueID', 'WordContext', 'image-label1', 'image-label1_more-edit', 'image-label2', 'image-label_N-ratings', 'image-label_ratings-per-image_mean', 'image-label_nameability_mean', 'image-label_nameability_SD', 'image-label_consistency_mean', 'image-label_consistency_SD', 'image-label_SE-nameability_mean', 'image-label_SE-consistency_mean', 'property_manmade_mean', 'property_manmade_SD', 'property_precious_mean', 'property_precious_SD', 'property_lives_mean', 'property_lives_SD', 'property_heavy_mean', 'property_heavy_SD', 'property_natural_mean', 'property_natural_SD', 'property_moves_mean', 'property_moves_SD', 'property_grasp_mean', 'property_grasp_SD', 'property_hold_me

In [108]:
log.info("Filtering …")
result = filter_concepts(concepts, params)


  INFO  Filtering …
  INFO  Memorability (memorability_cr) (0.7, 1.0) → 22759/27961 kept
  INFO  Recognizability (recognizability) (0.7, 1.0) → 7298/22759 kept
  INFO  Holdability (property_hold_mean) (5.0, 7.0) → 4207/7298 kept
  INFO  Pleasantness (property_pleasant_mean) (4.0, 6.0) → 3332/4207 kept
  INFO  Arousal (property_arousal_mean) (1.0, 4.5) → 2981/3332 kept


In [117]:
def dedupe():
    """
    Deduplicate words by keeping only the most common meaning/synset for each word.
    This is done by finding the concept with the highest SUBTLEX frequency for each
    unique word, and then calculating the standard Zipf scale for those concepts.
    """
    if Col.WORD not in result.columns or Col.SUBTLEX_FREQ not in result.columns:
        log.warning("Required columns 'Word' or 'SUBTLEX' missing — skipping deduplication.")
        return

    df = result.copy()

    # ===== 1. Handle missing values for safe grouping =====
    # Fill missing SUBTLEX values with 0 in a temporary series to avoid group idxmax crashes
    temp_subtlex = df[Col.SUBTLEX_FREQ].fillna(0.0)

    # ===== 2. Dedupe concepts to one row per unique word =====
    # Find the row index of the highest frequency concept for each word.
    # Passing df[Col.WORD] directly avoids Series key alignment errors.
    idx = temp_subtlex.groupby(df[Col.WORD]).idxmax()

    # Retrieve the full rows for our chosen indexes
    unique_df = df.loc[idx].reset_index(drop=True)
    print(f"Deduped to {len(unique_df)} unique words")

    # ===== 3. Handle missing values for final Zipf calculation =====
    # Log any words that had NaNs in SUBTLEX frequency before dropping them
    missing_freq = unique_df[unique_df[Col.SUBTLEX_FREQ].isna()]
    if len(missing_freq):
        print(f"{len(missing_freq)} words with no SUBTLEX freq value: {missing_freq[Col.WORD].tolist()}")

    # Drop any remaining NaNs in the frequency column
    clean_df = unique_df.dropna(subset=[Col.SUBTLEX_FREQ]).reset_index(drop=True)

    # ===== 4. Convert raw SUBTLEX frequency -> Zipf scale =====
    # Standard Zipf formula (van Heuven et al., 2014): Zipf = log10(freq_per_million) + 3
    # SUBTLEX-US corpus size = ~51 million words.
    CORPUS_SIZE_MILLIONS = 51.0
    
    # Calculate freq per million and clip to avoid taking log of absolute zero
    clean_df["SUBTLEX_per_million"] = clean_df[Col.SUBTLEX_FREQ] / CORPUS_SIZE_MILLIONS
    clean_df[Col.ZIPF_SUBTLEX] = np.log10(clean_df["SUBTLEX_per_million"].clip(lower=0.01)) + 3
    
    # Apply SUBTLEX convention floor: Words with 0 raw frequency default to Zipf 1.0
    clean_df.loc[clean_df[Col.SUBTLEX_FREQ] == 0, Col.ZIPF_SUBTLEX] = 1.0 

    # Sort final DataFrame descending by frequency for downstream pipeline consistency
    clean_df = clean_df.sort_values(Col.SUBTLEX_FREQ, ascending=False).reset_index(drop=True)

    print("\nZipf Score Summary Statistics:")
    print(clean_df[Col.ZIPF_SUBTLEX].describe())

    return clean_df

In [113]:
def report(df: pd.DataFrame) -> None:
    sep = "═" * 72
    print(f"\n{sep}\n  ✔  {len(df)} concepts match all filters\n{sep}")

    present = [c for c in DISPLAY_COLS if c in df.columns]

    if len(df):
        print(df[present].to_string(index=True))

    stat_cols = [c for c in STAT_COLS if c in df.columns]

    if stat_cols:
        df[stat_cols].describe().round(3).to_csv("descriptive-stats.csv")

    if "categories_53" in df.columns and len(df):
        df.explode("categories_53")["categories_53"].value_counts().to_csv("concepts-per-category.csv")


In [118]:
if params.output:
    # Save every DISPLAY_COLS column plus uniqueID.
    # categories_53 is a Python list — swap it for the CSV-safe semicolon string.
    save_cols = [_UNIQUE_ID] + [
        "cat53_string" if c == "categories_53" else c
        for c in DISPLAY_COLS
    ]

    missing = [c for c in save_cols if c not in result.columns]
    if missing:
        log.warning("Columns missing from result, skipped in output: %s", missing)


    result = dedupe()

    save_cols = [c for c in save_cols if c in result.columns]
    result[save_cols].to_csv(params.output, index=False)
    log.info("Saved -> %s  (%d rows x %d cols)", params.output, len(result), len(save_cols))

    report(result)


  WARNING  Columns missing from result, skipped in output: ['ZipfSUBTLEX_en']
  INFO  Saved -> results.csv  (469 rows x 22 cols)


Deduped to 471 unique words
2 words with no SUBTLEX freq value: ['lightning bug', 'rubber band']

Zipf Score Summary Statistics:
count   469.000
mean      3.127
std       1.228
min       1.000
25%       2.439
50%       3.342
75%       4.038
max       6.720
Name: ZipfSUBTLEX_en, dtype: float64

════════════════════════════════════════════════════════════════════════
  ✔  469 concepts match all filters
════════════════════════════════════════════════════════════════════════
                 Word                                                                                categories_53  size_mean  memorability_cr  recognizability  property_hold_mean  property_hold_SD  property_pleasant_mean  property_pleasant_SD  property_arousal_mean  property_arousal_SD  property_manmade_mean  property_manmade_SD  property_grasp_mean  property_grasp_SD  property_natural_mean  property_natural_SD  nameability  nameability_SE  SUBTLEX freq  ZipfSUBTLEX_en
0                 can                           